# Task 4.1b / 4.1c. Thực nghiệm GCN

| | |
|---|---|
| **Mục đích** | 4.1b: tinh chỉnh GCN-A; 4.1c: đánh giá GCN-B với L1 đã cố định |
| **Đầu vào** | Dataset `DiverseVul 9077 - Processed Graph Data`; Task 4.1c dùng version 1 |
| **Đầu ra** | 4.1b: `results_4_1b_gcn_tuning/`, `results_4_1b_gcn_final/`; 4.1c: `results_4_1c_gcn_graph_b/` |
| **Máy chạy** | Kaggle GPU, bật Internet |

Hai protocol `project_wise`, `seen_project`; seed 42, 43, 44. Giữ nguyên dữ liệu, vocabulary, split và 23 nhãn CWE. Chọn cấu hình và checkpoint bằng validation; test chỉ dùng để báo cáo.


In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
RUN_GRAPH_B = Path("/kaggle/input").is_dir()  # True trên Kaggle; False khi kiểm tra offline; đặt False để dùng Graph A
APPROVED_BRANCH = "feature/4_1b_gcn_tuning"
APPROVED_COMMIT = "5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5" if RUN_GRAPH_B else ""  # shared trainer SHA
DATASET_ID = "vtnmai/diversevul-9077-processed-graph-data/versions/1" if RUN_GRAPH_B else ""
APPROVED_CONFIG_IDS = []  # Danh sách đã thống nhất: B0, E1, D1, W1, H1, L1, N1
RUN_TUNING = False
RUN_FINAL_TEST = False
# Để phục hồi hoặc chỉ đánh giá test: trỏ tới output đã lưu, có frozen_selection.json.
SAVED_RESULTS_ROOT = None  # ví dụ /kaggle/input/<saved-output>/results_4_1b_gcn_tuning
OUT_DIR = Path("/kaggle/working/results_4_1b_gcn_tuning")
FINAL_OUT_DIR = Path("/kaggle/working/results_4_1b_gcn_final")
GRAPH_B_ROOT = Path("/kaggle/working/results_4_1c_gcn_graph_b")
GRAPH_B_OUT_DIR = GRAPH_B_ROOT / "L1"
if RUN_GRAPH_B and (RUN_TUNING or RUN_FINAL_TEST or SAVED_RESULTS_ROOT is not None):
    raise ValueError("Graph B phải chạy riêng, không dùng tuning/final/checkpoint Graph A.")
ACTIVE = RUN_TUNING or RUN_FINAL_TEST or RUN_GRAPH_B
if ACTIVE:
    if not Path("/kaggle/input").is_dir():
        raise RuntimeError("Chỉ chạy trên Kaggle với dataset chính thức.")
    if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
        raise ValueError("Cần SHA đầy đủ đã được phê duyệt.")
    if not re.fullmatch(r"vtnmai/diversevul-9077-processed-graph-data/versions/[1-9][0-9]*", DATASET_ID):
        raise ValueError("Ghi phiên bản Kaggle dataset đã gắn vào notebook.")
REPO_DIR = Path("/tmp/g41c" if RUN_GRAPH_B else "/tmp/g41b") / APPROVED_COMMIT
READ_ROOT = Path(SAVED_RESULTS_ROOT) if SAVED_RESULTS_ROOT else OUT_DIR


## 1. Mã nguồn và môi trường

Task 4.1c tự bật trên Kaggle khi có `/kaggle/input`; hai cờ Graph A giữ `False`. Mã train lấy từ nhánh đã công bố `feature/4_1b_gcn_tuning`, checkout SHA `5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5`. Notebook upload chưa commit được ghi provenance riêng. Lựa chọn L1 lấy từ Git object của delivery commit 4.1b và kiểm tra SHA-256.

Bật GPU, Internet và gắn dataset processed **version 1**. Phiên bản dataset phải được chọn đúng trên Kaggle; giá trị cấu hình chỉ ghi nhận danh tính. Repository tạm ở `/tmp`, ngoài thư mục output. Lưu notebook đã thực thi và Saved Version cùng kết quả.

Để chạy Graph A, đặt `RUN_GRAPH_B=False`, điền SHA và dataset version phù hợp, sau đó chọn cờ tuning hoặc final như các mục dưới. Không cần chờ merge vào `main`.


In [ ]:
if ACTIVE:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", "--single-branch", "--branch",
                        APPROVED_BRANCH, REPO_URL, str(REPO_DIR)], check=True, timeout=180)
    subprocess.run(["git", "checkout", "--detach", APPROVED_COMMIT], cwd=REPO_DIR, check=True)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    assert actual == APPROVED_COMMIT
    if RUN_GRAPH_B:
        assert actual == "5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5", "Sai shared trainer SHA."
    sys.path.insert(0, str(REPO_DIR))
    import torch
    from src.training.gcn_tuning import (
        BASELINE, CANDIDATES, HYPOTHESES, PROTOCOLS, SEEDS,
        configuration, code_identity, environment, freeze_winner, write_json,
        reuse_candidate, read_frozen, validate_context, validate_final,
        digest, data_identity, validate_artifacts,
    )
    from src.graph.processed import ids_hash, load_schema
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings."
    assert torch.ones(1, device="cuda").sum().item() == 1, "Không khởi tạo được GPU."
    assert not code_identity(REPO_DIR)["dirty"], "Checkout phải sạch."
    print("Revision:", actual, "Environment:", environment(), flush=True)


def run_script(arguments, log_path):
    cmd = [sys.executable, "scripts/train_gnn.py", *arguments]
    print("Command:", cmd, flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("x", encoding="utf-8") as handle:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,
                                env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        if proc.wait():
            raise RuntimeError(f"Script thất bại; giữ output và xem {log_path}")


## 2. Dữ liệu và cấu hình

Kiểm tra manifest, vocabulary, 23 nhãn và số mẫu của từng split. Task 4.1b so sánh B0 với sáu biến thể, mỗi biến thể đổi một yếu tố: E1 tiêu chí dừng, D1 dropout, W1 weight decay, H1 dim, L1 số lớp, N1 giới hạn node. Chọn theo Macro-F1 validation trung bình của `project_wise`; `seen_project` là đối chứng. Thống nhất danh sách cấu hình trước khi chạy. Task 4.1c chỉ dùng L1 đã chọn.


In [ ]:
if ACTIVE:
    found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
             if all((p.parent / protocol / "vocab.json").is_file() for protocol in PROTOCOLS)]
    assert len(found) == 1, "Gắn đúng một phiên bản dataset processed chính thức."
    PROCESSED = found[0]
    metadata = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    assert metadata.get("spec_version") in {"1.1", "1.2"}
    assert metadata.get("n_valid_samples") == 7799
    assert sorted(load_schema(REPO_DIR / "configs/graph/schema_spec.yaml")["cwe_mapping"].values()) == list(range(23))
    from collections import Counter
    expected = {"project_wise": {"train": 6101, "validation": 762, "test": 936},
                "seen_project": {"train": 6272, "validation": 764, "test": 763}}
    for protocol in PROTOCOLS:
        with (PROCESSED / protocol / "manifest.csv").open(encoding="utf-8", newline="") as handle:
            manifest = list(csv.DictReader(handle))
        counts = Counter(row["split"] for row in manifest)
        assert len({row["sample_id"] for row in manifest}) == len(manifest)
        assert all(0 <= int(row["y"]) < 23 for row in manifest)
        vocab = json.loads((PROCESSED / protocol / "vocab.json").read_text(encoding="utf-8"))
        assert isinstance(vocab["k"], int) and vocab["k"] > 0
        for name in ("node_type", "token"):
            assert sorted(vocab[name].values()) == list(range(len(vocab[name])))
            assert vocab[name].get("<PAD>") == 0 and vocab[name].get("<UNK>") == 1
        assert vocab["token"].get("<MASK>") == 2
        assert vocab.get("train_ids_sha256") == ids_hash(row["sample_id"] for row in manifest if row["split"] == "train")
        assert dict(counts) == expected[protocol]
        for split in expected[protocol]:
            graph = "graph_b" if RUN_GRAPH_B else "graph_a"
            assert (PROCESSED / protocol / f"{graph}_{split}.pt").is_file()
    print("Dataset:", DATASET_ID, PROCESSED)
if RUN_TUNING:
    assert APPROVED_CONFIG_IDS and len(set(APPROVED_CONFIG_IDS)) == len(APPROVED_CONFIG_IDS)
    assert all(key in CANDIDATES for key in APPROVED_CONFIG_IDS)
    for key in APPROVED_CONFIG_IDS:
        cfg = configuration(key)
        changed = {k: v for k, v in cfg.items() if v != BASELINE[k]}
        assert len(changed) == (0 if key == "B0" else 1)
        print(key, changed, HYPOTHESES[key], flush=True)
    print("Số cấu hình:", len(APPROVED_CONFIG_IDS), "Số lần train tối đa:", 6 * len(APPROVED_CONFIG_IDS))


## 3. Huấn luyện Graph A trên train/validation

`RUN_TUNING=True` chạy các cấu hình đã duyệt, không nạp test. Kết quả hoàn tất chỉ được tái sử dụng khi artifact, cấu hình, mã, dữ liệu và môi trường đều khớp. `SAVED_RESULTS_ROOT` cho phép lấy lại output đã lưu; không resume optimizer. Giữ nguyên kết quả dở dang để kiểm tra và dùng thư mục mới khi cần chạy lại.


In [ ]:
if RUN_TUNING:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    plan = {
        "code": code_identity(REPO_DIR), "dataset_id": DATASET_ID,
        "configurations": {key: configuration(key) for key in APPROVED_CONFIG_IDS},
        "hypotheses": {key: HYPOTHESES[key] for key in APPROVED_CONFIG_IDS},
        "protocols": list(PROTOCOLS), "seeds": list(SEEDS),
        "selection": "project_wise.validation.macro_f1.mean",
        "checkpoint_selection": "maximum raw validation Macro-F1; earliest tie",
    }
    plan_path = OUT_DIR / "experiment_plan.json"
    if plan_path.exists():
        if json.loads(plan_path.read_text(encoding="utf-8")) != plan:
            raise ValueError("Kế hoạch đã lưu khác cấu hình hiện tại; không ghi đè.")
    else:
        with plan_path.open("x", encoding="utf-8") as handle:
            json.dump(plan, handle, ensure_ascii=False, indent=2)
    for key in APPROVED_CONFIG_IDS:
        target = OUT_DIR / key
        source = target if target.exists() else READ_ROOT / key
        if source.exists():
            reuse_candidate(source, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID)
            if source.resolve() != target.resolve():
                shutil.copytree(source, target)  # destination must be unused
        else:
            args = ["--processed-dir", str(PROCESSED), "--out-dir", str(target),
                    "--models", "gcn", "--graphs", "graph_a", "--protocols", *PROTOCOLS,
                    "--seeds", *map(str, SEEDS), "--validation-only", "--save-checkpoints",
                    "--config-id", key, "--dataset-id", DATASET_ID]
            for name, value in configuration(key).items():
                if value is not None:
                    args += ["--" + name.replace("_", "-"), str(value)]
            run_script(args, OUT_DIR / f"{key}.log")
            reuse_candidate(target, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID)


## 4. Cố định cấu hình Graph A

Chọn bằng Macro-F1 validation trung bình chưa làm tròn của `project_wise`, cùng một cấu hình cho hai protocol. Khi bằng nhau, ưu tiên B0, E1, D1, W1, H1, L1, N1. Lựa chọn đã lưu chỉ được xác minh, không ghi đè hoặc chọn lại theo test.


In [ ]:
if RUN_TUNING or RUN_FINAL_TEST:
    schema = REPO_DIR / "configs/graph/schema_spec.yaml"
    FROZEN_ROOT = OUT_DIR if RUN_TUNING else READ_ROOT
    FROZEN = FROZEN_ROOT / "frozen_selection.json"
    saved_frozen = READ_ROOT / "frozen_selection.json"
    if RUN_TUNING and not FROZEN.exists() and saved_frozen.exists():
        previous = read_frozen(saved_frozen, REPO_DIR, PROCESSED, schema, READ_ROOT)
        if {r["config_id"] for r in previous["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không tự chọn lại.")
        with FROZEN.open("xb") as handle:
            handle.write(saved_frozen.read_bytes())
    if FROZEN.exists():
        frozen = read_frozen(FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
        if RUN_TUNING and {r["config_id"] for r in frozen["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không ghi đè.")
    elif RUN_TUNING:
        frozen = freeze_winner([OUT_DIR / key for key in APPROVED_CONFIG_IDS], FROZEN)
    else:
        raise FileNotFoundError(f"Thiếu lựa chọn validation đã cố định: {FROZEN}")
    validate_context(frozen["winner"]["metadata"], REPO_DIR, PROCESSED, schema, DATASET_ID)
    print("Validation ranking:", frozen["ranking"], flush=True)
    print("Frozen configuration:", frozen["winner"]["config_id"], flush=True)


## 5. Đánh giá test Graph A

Trong phiên riêng, gắn dataset và output tuning đã lưu; đặt `RUN_TUNING=False`, `RUN_FINAL_TEST=True`, `SAVED_RESULTS_ROOT` và cùng SHA/version. Nạp checkpoint đã chọn, không train lại. Output final phải riêng với tuning; kết quả hoàn tất hợp lệ được tái sử dụng, kết quả dở dang được giữ để kiểm tra.


In [ ]:
if RUN_FINAL_TEST:
    if FINAL_OUT_DIR.exists() and any(FINAL_OUT_DIR.iterdir()):
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
    else:
        run_script(["--processed-dir", str(PROCESSED), "--out-dir", str(FINAL_OUT_DIR),
                    "--frozen-selection", str(FROZEN), "--results-root", str(FROZEN_ROOT)],
                   FINAL_OUT_DIR.parent / (FINAL_OUT_DIR.name + ".log"))
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
    print((FINAL_OUT_DIR / "summary.md").read_text(encoding="utf-8"))


## 6. Task 4.1c — GCN-B với L1

`RUN_GRAPH_B=True` chạy riêng sáu lần train mới và sáu lần đánh giá test, không tìm kiếm cấu hình hoặc nạp trọng số Graph A. L1 được đọc từ frozen-selection lịch sử và kiểm tra SHA-256. Checkpoint có Macro-F1 validation chưa làm tròn cao nhất được giữ; tie chọn epoch sớm nhất. Pipeline nạp tensor test trước train nhưng chỉ đánh giá sau khi chọn checkpoint.

`code` ghi SHA shared trainer; `notebook_provenance` ghi riêng notebook upload chưa commit và băm lịch sử cell. `experiment_plan.json` lưu cấu hình, provenance và chính sách đánh giá. Output phải mới; giữ kết quả dở dang để kiểm tra, không tự resume hoặc ghi đè. Confusion matrix được xuất cho seed 42; summary giữ tiêu đề lịch sử Task 4.1a của exporter chung.


In [ ]:
if RUN_GRAPH_B:
    assert DATASET_ID == "vtnmai/diversevul-9077-processed-graph-data/versions/1"
    assert list(PROTOCOLS) == ["project_wise", "seen_project"] and list(SEEDS) == [42, 43, 44]
    import hashlib
    frozen_reference = "e2ccb58def76ca3f3fd329b17627cbff924dd914:reports/graph/4_1b_results/tuning/frozen_selection.json"
    frozen_sha = "47b7ae96a1918813aa37b36416492ce355045a8001a794857bc3a4bf9fa88900"
    frozen_bytes = subprocess.check_output(["git", "show", frozen_reference], cwd=REPO_DIR)
    assert hashlib.sha256(frozen_bytes).hexdigest() == frozen_sha, "Frozen selection 4.1b đã thay đổi."
    reference = json.loads(frozen_bytes)
    winner = reference["winner"]
    previous = winner["metadata"]["arguments"]
    assert reference["format_version"] == 2
    assert reference["selection_metric"] == "project_wise.validation.macro_f1.mean"
    assert winner["config_id"] == previous["config_id"] == "L1"
    assert previous["dataset_id"] == DATASET_ID and previous["limit"] is None
    assert previous["protocols"] == list(PROTOCOLS) and previous["seeds"] == list(SEEDS)
    graph_b_cfg = {name: previous[name] for name in BASELINE}
    assert graph_b_cfg == configuration("L1"), "L1 hiện tại khác cấu hình đã cố định."
    assert graph_b_cfg["max_nodes"] is None
    assert GRAPH_B_ROOT.is_absolute()
    for historical in (OUT_DIR, FINAL_OUT_DIR):
        assert not (GRAPH_B_ROOT.resolve().is_relative_to(historical.resolve())
                    or historical.resolve().is_relative_to(GRAPH_B_ROOT.resolve()))
    graph_b_args = ["--processed-dir", str(PROCESSED), "--out-dir", str(GRAPH_B_OUT_DIR),
                    "--models", "gcn", "--graphs", "graph_b", "--protocols", *PROTOCOLS,
                    "--seeds", *map(str, SEEDS), "--save-checkpoints",
                    "--config-id", "L1", "--dataset-id", DATASET_ID]
    for name, value in graph_b_cfg.items():
        if value is not None:
            graph_b_args += ["--" + name.replace("_", "-"), str(value)]
    graph_b_plan = {
        "task": "4.1c", "code": code_identity(REPO_DIR), "environment": environment(),
        "dataset_id": DATASET_ID, "processed_dir": str(PROCESSED),
        "graph": "graph_b", "model": "gcn", "config_id": "L1", "configuration": graph_b_cfg,
        "protocols": list(PROTOCOLS), "seeds": list(SEEDS), "limit": None,
        "training_runs": 6, "test_evaluations": 6,
        "notebook_provenance": {
            "origin": "direct Kaggle upload of uncommitted Task 4.1c notebook",
            "source_commit": None,
            "execution_inputs_sha256": hashlib.sha256(json.dumps(
                get_ipython().history_manager.input_hist_raw, ensure_ascii=False).encode("utf-8")).hexdigest(),
            "archive": "Save executed notebook and Kaggle Saved Version with outputs",
        },
        "source_branch": APPROVED_BRANCH,
        "frozen_selection_reference": frozen_reference,
        "frozen_selection_sha256": frozen_sha, "selection_metric": reference["selection_metric"],
        "original_selection_score": winner["score"],
        "original_experiment_code": winner["metadata"]["code"],
        "checkpoint_selection": "maximum raw validation Macro-F1; earliest tie",
        "test_policy": "evaluate once per validation-selected model; never select using test",
        "schema_sha256": digest(REPO_DIR / "configs/graph/schema_spec.yaml"),
        "inputs": {p: data_identity(PROCESSED, p, "graph_b") for p in PROTOCOLS},
        "test_sha256": {p: digest(PROCESSED / p / "graph_b_test.pt") for p in PROTOCOLS},
        "output_dir": str(GRAPH_B_OUT_DIR), "arguments": graph_b_args,
    }


### 6.1. Thực thi và kiểm tra kết quả

Sau khi kiểm tra cấu hình và lệnh, chạy cell dưới để lưu kế hoạch, thực thi script chung và xác minh artifact. So sánh với GCN-A L1, mặc định 4.1a và token baseline. Khi diễn giải, nêu giới hạn `seen_project`: 5 mẫu validation và 6 mẫu test trùng input train.


In [ ]:
if RUN_GRAPH_B:
    if GRAPH_B_ROOT.exists() and any(GRAPH_B_ROOT.iterdir()):
        raise FileExistsError(f"Giữ nguyên kết quả cũ; chọn GRAPH_B_ROOT mới: {GRAPH_B_ROOT}")
    GRAPH_B_ROOT.mkdir(parents=True, exist_ok=True)
    with (GRAPH_B_ROOT / "experiment_plan.json").open("x", encoding="utf-8") as handle:
        json.dump(graph_b_plan, handle, ensure_ascii=False, indent=2, allow_nan=False)
    run_script(graph_b_args, GRAPH_B_ROOT / "graph_b.log")
    graph_b_meta = validate_artifacts(GRAPH_B_OUT_DIR)
    assert graph_b_meta["code"] == graph_b_plan["code"]
    assert graph_b_meta["environment"] == graph_b_plan["environment"]
    recorded = graph_b_meta["arguments"]
    assert all(recorded[k] == v for k, v in graph_b_cfg.items())
    assert recorded["graphs"] == ["graph_b"] and recorded["models"] == ["gcn"]
    assert recorded["config_id"] == "L1" and recorded["dataset_id"] == DATASET_ID
    assert recorded["protocols"] == list(PROTOCOLS) and recorded["seeds"] == list(SEEDS)
    assert recorded["save_checkpoints"] and not recorded["validation_only"]
    assert recorded["frozen_selection"] is None and recorded["limit"] is None
    with (GRAPH_B_OUT_DIR / "results.csv").open(encoding="utf-8", newline="") as handle:
        graph_b_rows = list(csv.DictReader(handle))
    assert len(graph_b_rows) == 12
    assert {(r["protocol"], int(r["seed"]), r["split"]) for r in graph_b_rows} == {
        (p, s, split) for p in PROTOCOLS for s in SEEDS for split in ("validation", "test")}
    assert all(r["graph"] == "graph_b" and r["model"] == "gcn" for r in graph_b_rows)
    runs = graph_b_meta["runs"]
    assert len(runs) == 6 and {(r["protocol"], r["seed"]) for r in runs} == {
        (p, s) for p in PROTOCOLS for s in SEEDS}
    for run in runs:
        assert run["checkpoint"] == f"{run['protocol']}_graph_b_gcn_{run['seed']}.pt"
        assert digest(GRAPH_B_OUT_DIR / run["checkpoint"]) == run["sha256"]
    print((GRAPH_B_OUT_DIR / "summary.md").read_text(encoding="utf-8"))
